# Notebook 6 — Trend Analysis & Forecasting
Detect revenue trends (UP/FLAT/DOWN) and forecast next year's revenue using linear regression and Holt-Winters. **Model estimates only — not financial advice.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120})

DATA = Path('../data/clean')
pl = pd.read_csv(DATA / 'profitandloss.csv')
co = pd.read_csv(DATA / 'companies.csv')
pl = pl[pl['is_ttm'] == False].copy()
print('Data loaded. Rows:', len(pl))

## Step 1 — Linear regression trend classification per company

In [ ]:
trend_records = []
for sym, grp in pl.groupby('company_id'):
    grp = grp.sort_values('fiscal_year')
    recent = grp[grp['fiscal_year'] >= grp['fiscal_year'].max() - 4]
    if len(recent) < 2:
        trend_records.append({'symbol': sym, 'slope': 0, 'trend': 'INSUFFICIENT_DATA', 'r2': None})
        continue
    x = np.arange(len(recent), dtype=float)
    y = recent['sales'].fillna(0).values
    coeffs = np.polyfit(x, y, 1)
    slope = coeffs[0]
    y_pred = np.polyval(coeffs, x)
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    avg_sales = y.mean()
    slope_pct = (slope / avg_sales * 100) if avg_sales != 0 else 0
    if slope_pct > 5:
        trend = 'UP'
    elif slope_pct < -5:
        trend = 'DOWN'
    else:
        trend = 'FLAT'
    trend_records.append({'symbol': sym, 'slope': round(slope, 2),
                          'slope_pct': round(slope_pct, 2), 'r2': round(r2, 3), 'trend': trend})

trend_df = pd.DataFrame(trend_records)
print(trend_df['trend'].value_counts())
print()
print(trend_df.sort_values('slope_pct', ascending=False).head(10).to_string(index=False))

## Step 2 — Trend distribution chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = trend_df['trend'].value_counts()
colors_t = {'UP':'#22c55e','FLAT':'#f59e0b','DOWN':'#ef4444','INSUFFICIENT_DATA':'#94a3b8'}
axes[0].bar(counts.index, counts.values,
            color=[colors_t.get(t,'steelblue') for t in counts.index], alpha=0.85)
axes[0].set_title('Revenue trend classification (5-year linear regression)')
axes[0].set_ylabel('Company count')

up_co = trend_df[trend_df['trend']=='UP'].nlargest(15,'slope_pct')
axes[1].barh(up_co['symbol'], up_co['slope_pct'], color='#22c55e', alpha=0.85)
axes[1].set_title('Top 15 companies by revenue slope % (UP trend)')
axes[1].set_xlabel('Annual slope as % of avg sales')
plt.tight_layout()
plt.show()

## Step 3 — Holt-Winters forecasting: top 20 by revenue

In [ ]:
latest_rev = pl.sort_values('fiscal_year').groupby('company_id')['sales'].last()
top20 = latest_rev.nlargest(20).index.tolist()

forecast_records = []
for sym in top20:
    grp = pl[pl['company_id']==sym].sort_values('fiscal_year')
    series = grp.set_index('fiscal_year')['sales'].dropna()
    if len(series) < 4:
        continue
    try:
        model = ExponentialSmoothing(series.values, trend='add', damped_trend=True)
        fit = model.fit(optimized=True, remove_bias=True)
        forecast = fit.forecast(1)[0]
        conf_lo = forecast * 0.9
        conf_hi = forecast * 1.1
    except Exception:
        slope = np.polyfit(np.arange(len(series)), series.values, 1)[0]
        forecast = series.values[-1] + slope
        conf_lo, conf_hi = forecast * 0.85, forecast * 1.15
    forecast_records.append({'symbol': sym, 'last_year': int(series.index[-1]),
                             'last_sales': round(series.values[-1], 2),
                             'forecast_year': int(series.index[-1]) + 1,
                             'forecast_sales': round(forecast, 2),
                             'conf_lo': round(conf_lo, 2), 'conf_hi': round(conf_hi, 2)})

forecast_df = pd.DataFrame(forecast_records)
print('⚠️  MODEL ESTIMATES ONLY — NOT FINANCIAL ADVICE')
print(forecast_df[['symbol','last_sales','forecast_sales','conf_lo','conf_hi']].to_string(index=False))

## Step 4 — Forecast charts with confidence intervals

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20, 14))
axes = axes.flatten()

for i, row in forecast_df.iterrows():
    if i >= len(axes):
        break
    ax = axes[i]
    sym = row['symbol']
    grp = pl[pl['company_id']==sym].sort_values('fiscal_year')
    series = grp.set_index('fiscal_year')['sales'].dropna()
    ax.plot(series.index, series.values/1000, marker='o', color='steelblue', markersize=3)
    ax.plot([row['last_year'], row['forecast_year']],
            [row['last_sales']/1000, row['forecast_sales']/1000],
            marker='*', color='green', linestyle='--', markersize=8)
    ax.fill_between([row['last_year'], row['forecast_year']],
                    [row['last_sales']/1000, row['conf_lo']/1000],
                    [row['last_sales']/1000, row['conf_hi']/1000],
                    alpha=0.15, color='green')
    ax.set_title(sym, fontsize=9)
    ax.set_ylabel('₹000 Cr', fontsize=7)
    ax.tick_params(labelsize=7)

for j in range(len(forecast_df), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Revenue forecasts — top 20 companies\n⚠️ Model estimates only, not financial advice',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Step 5 — Export trend labels and forecasts

In [ ]:
trend_df.to_csv('../data/clean/trend_labels.csv', index=False)
forecast_df.to_csv('../data/clean/revenue_forecasts.csv', index=False)
print('Exported:')
print('  data/clean/trend_labels.csv    —', len(trend_df), 'companies')
print('  data/clean/revenue_forecasts.csv —', len(forecast_df), 'companies')
print()
print('⚠️  All forecasts are model estimates only and should not be used as financial advice.')